In [ ]:
# CSE 4310 
# Programming Assignment 3 (P3)

In [ ]:
'''
Alex Nguyen
1002097417
'''

### Task 1 (30 points) 

Implement Hough transform from scratch to detect circles in the image. Refer to this wiki page for pseudocode(https://en.wikipedia.org/wiki/Circle_Hough_Transform). Superimpose detected circles on the original image.

Note: You shouldn't use cv2.HoughCircles() to detect the circles.

<img src = Images/coins.png>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

importedCoins = cv2.imread("Images/coins.png", cv2.IMREAD_GRAYSCALE)

In [ ]:
# We first need to do edge detection. This will be so much better now since I can just directly call cv2.Canny
blurred_coins = cv2.GaussianBlur(importedCoins, (5, 5), 1)

edges = cv2.Canny(blurred_coins, 50, 150)

# plt.imshow(edges, cmap="gray") 
# Checked to see that the Canny Edge Detection worked :D

In [ ]:
# Now even though this is slow, I will make r a parameter, so thus, our accumulator array will be 3 dimensional. 
# The reason that I'm doing this is because I am too lazy to do fine-tuning and finding all the radius for the respective coins.

# Firstly I need to make the accumulator array.
height, width = edges.shape

r_min = 10
r_max = 100

A = np.zeros(
    (height, width, r_max - r_min + 1),
    dtype=np.uint32
)

angles = np.deg2rad(np.arange(0, 360))
# Obtain all edge coordinates
ys, xs = np.nonzero(edges)

# Originally this used no gradients, but since there is a lot of calculations, so we will use those here.
# ... which now means I have to calculate the direction vector
Gx = cv2.Sobel(blurred_coins, cv2.CV_32F, 1, 0, ksize=3)
Gy = cv2.Sobel(blurred_coins, cv2.CV_32F, 0, 1, ksize=3)

# Gradient vectors at the edge pixels
gx = Gx[ys, xs]
gy = Gy[ys, xs]

magnitude = np.sqrt(gx**2 + gy**2)

# Avoid division by zero (cool way AI showed me to do this)
valid = magnitude > 1e-6

xs = xs[valid]
ys = ys[valid]
gx = gx[valid]
gy = gy[valid]
magnitude = magnitude[valid]

# Normalize the gradient vectors
nx = gx / magnitude
ny = gy / magnitude

# Now we populate the accumulator array. "radius_small and radius_big"
for index, r in enumerate(range(r_min, r_max + 1)):
    for direction in [-1, 1]: # This increments a point in the direction of the normalized vector (backwards and forwards)

        a = np.rint(xs + direction * r * nx).astype(int)
        b = np.rint(ys + direction * r * ny).astype(int)

        valid_centers = (
            (a >= 0) & (a < width) &
            (b >= 0) & (b < height)
        )

        np.add.at(
            A[:, :, index],
            (b[valid_centers], a[valid_centers]),
            1
        )

In [ ]:
# Performance of non-max suppression to localize circles better

# Thank you AI for this implementation
def nms_hough_circles(accumulator, threshold=15, num_circles=10,
                      center_distance=20, radius_distance=10):

    # Make a copy so we don't modify the original accumulator
    A_copy = accumulator.copy()

    detected_circles = []
    votes = []

    height, width, radius_count = A_copy.shape

    for _ in range(num_circles):

        # Find the highest vote in the entire accumulator
        flat_index = np.argmax(A_copy)

        y, x, r_index = np.unravel_index(
            flat_index, A_copy.shape
        )

        max_votes = A_copy[y, x, r_index]

        # Stop if the remaining candidates are too weak
        if max_votes < threshold:
            break

        detected_circles.append((y, x, r_index))
        votes.append(max_votes)

        # Suppress candidates near this detected circle
        y_min = max(0, y - center_distance)
        y_max = min(height, y + center_distance + 1)

        x_min = max(0, x - center_distance)
        x_max = min(width, x + center_distance + 1)

        r_min_idx = max(0, r_index - radius_distance)
        r_max_idx = min(radius_count, r_index + radius_distance + 1)

        A_copy[
            y_min:y_max,
            x_min:x_max,
            r_min_idx:r_max_idx
        ] = 0

    return np.array(detected_circles).reshape(-1, 3), np.array(votes)

In [ ]:
# Perform non-maximum suppression
circles, votes = nms_hough_circles(
    A,
    threshold=15,
    num_circles=10,
    center_distance=20,
    radius_distance=10
)

# Select the 10 highest-voted candidates
top_10_circles = circles[:10]
top_10_votes = votes[:10]

# Display detected circle parameters
for rank, ((y, x, r_index), vote) in enumerate(
    zip(top_10_circles, top_10_votes), start=1
):
    radius = r_index + r_min

    print(
        f"Rank {rank}: "
        f"Center ({x}, {y}), "
        f"Radius {radius}, "
        f"Votes {vote}"
    )

# Superimpose circles on the original image
from matplotlib.patches import Circle

fig, ax = plt.subplots(figsize=(10, 8))

ax.imshow(importedCoins, cmap="gray")

for y, x, r_index in top_10_circles:

    radius = r_index + r_min

    circle = Circle(
        (x, y),
        radius=radius,
        facecolor="none",
        edgecolor="red",
        linewidth=2
    )

    ax.add_patch(circle)

ax.set_title("Task 1")
ax.set_aspect("equal")

plt.show()

### Task 2 (30 points)

Implement Harris corner detector from scratch. Mark detected corners on original image. 

Note: You shouldn't use cv2.cornerHarris() to detect the corners.

<img src = Images/checkerboard.jpg>

In [ ]:
# We start with importing as usual
imported_checkerboard = cv2.imread("Images/checkerboard.jpg", cv2.IMREAD_GRAYSCALE)

# Now for the harris detection, we gotta go into the slides real quick

In [ ]:
# Step 1: Compute image gradients X and Y respeectively
I_x = cv2.Sobel(imported_checkerboard, ddepth=cv2.CV_64F, dx=1, dy=0, ksize=5, scale=1, delta=0, borderType=cv2.BORDER_DEFAULT)
I_y = cv2.Sobel(imported_checkerboard, ddepth=cv2.CV_64F, dx=0, dy=1, ksize=5, scale=1, delta=0, borderType=cv2.BORDER_DEFAULT)

In [ ]:
# Step 2: Compute products of derivatives, Ix^2, Iy^2, Ixy 
I_xx = I_x ** 2
I_yy = I_y ** 2
I_xy = I_x * I_y

In [ ]:
# Step 3: Do the guass blur thing
S_xx = cv2.GaussianBlur(I_xx, (5, 5), sigmaX=1)
S_yy = cv2.GaussianBlur(I_yy, (5, 5), sigmaX=1)
S_xy = cv2.GaussianBlur(I_xy, (5, 5), sigmaX=1)

In [ ]:
# Step 4: Build the M matrix thing

# Cool AI implementation better than my for loops
M = np.zeros((*S_xx.shape, 2, 2), dtype=np.float64)

M[:, :, 0, 0] = S_xx
M[:, :, 0, 1] = S_xy
M[:, :, 1, 0] = S_xy
M[:, :, 1, 1] = S_yy

In [ ]:
# Step 5: Compute the response of the detector at each pixel
k = 0.04

det_M = np.linalg.det(M)
trace_M = np.trace(M, axis1=2, axis2=3)

R = det_M - k * (trace_M ** 2)

In [ ]:
# Step 6: Nonmax and thresholding

# Thresholding:

threshold = 0.01 * R.max()

R_thresholded = np.where(R > threshold, R, 0)

In [ ]:
# Nonmax:

height, width = R_thresholded.shape

R_nms = np.zeros_like(R_thresholded)

window_size = 3
offset = window_size // 2

for r in range(offset, height - offset):
    for c in range(offset, width - offset):

        current = R_thresholded[r, c]

        if current <= 0:
            continue

        neighborhood = R_thresholded[
            r - offset:r + offset + 1,
            c - offset:c + offset + 1
        ]

        if current == np.max(neighborhood):
            R_nms[r, c] = current

In [ ]:

# Displaying time
corner_rows, corner_cols = np.where(R_nms > 0)

output = cv2.cvtColor(imported_checkerboard, cv2.COLOR_GRAY2BGR)

for r, c in zip(corner_rows, corner_cols):
    cv2.circle(output, (int(c), int(r)), 3, (0, 0, 255), -1)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.title("Task 2")
plt.show()

### Task 3 (40 points)

Implement multiscale template matching that measures the SSD differences (normalized for brightness but NOT for contrast) between the template and each location of the image at multiple scales. Both image and template are grayscale images. For this task, consider a linear range of scales, np.arange( 0.5, 3, .25). Show your best match for each size(0.5, 0.75, 1 etc.) in a 2X5 plot. 

Template: <img src = Images/template.png>

Test Image: <img src = Images/test_image.png style="width:537px; height:400px">

Hint: function multiscale_correlation (in the slides) has many similarities with the function that you are asked to implement. The only difference is that multiscale_correlation computes scores based on normalized cross-correlation, whereas your scores are based on SSD differences after brightness normalization.

Note: 
1. You should implement template matching from scratch, and not use the opencv template matching.
2. Only use "bilinear interpolation" when resizing.

## Submission Guidelines:

    Submit through Canvas your source code in a single .ipynb file. The name of the .ipynb file should be YourStudentID.ipynb. (For example: 1001234567.ipynb)
    Import all the images from the ./Images directory. Your TA will use the same directory name to grade your submission.
    You don't need to attach the image file with your submission.